In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Install and Import Libraries**

In [2]:
!pip install transformers wandb -q

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

print("PyTorch:", torch.__version__)
print("Device will be: CPU")

PyTorch: 2.10.0+cu128
Device will be: CPU


# **Configuration**

In [3]:
DEVICE      = "cpu"
DATA_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge"
MODEL_NAME  = "distilbert-base-uncased"
OPTION_COLS = ["A", "B", "C", "D", "E"]
MAX_LEN     = 64
BATCH_SIZE  = 32
EPOCHS      = 3
LR          = 3e-5
SEED        = 42
VAL_SIZE    = 0.1

print("Device:", DEVICE)
print("Model :", MODEL_NAME)

Device: cpu
Model : distilbert-base-uncased


# **W&B Setup**

In [4]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb.init(
    project="smart-mcq-solver",
    name="model3-deberta-v2-xlarge",
    config={
        "model_name" : MODEL_NAME,
        "max_len"    : MAX_LEN,
        "batch_size" : BATCH_SIZE,
        "epochs"     : EPOCHS,
        "lr"         : LR
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


# **Load Dataset**

In [5]:
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train:", train.shape)
print("Test :", test.shape)
train.head(3)

Train: (2000, 8)
Test : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


# **Train Validation Split**

In [6]:
torch.manual_seed(SEED)
np.random.seed(SEED)

train_df, val_df = train_test_split(
    train, test_size=VAL_SIZE, random_state=SEED, stratify=train['answer']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)

Train: (1800, 8)
Val  : (200, 8)


# **MAP@3 Utility**

In [7]:
def map_at_3(ground_truth, predictions):
    score = 0.0
    for k, pred in enumerate(predictions[:3], start=1):
        if pred == ground_truth:
            score = 1.0 / k
            break
    return score

def evaluate_map3(df, pred_col='prediction'):
    scores = []
    for _, row in df.iterrows():
        preds = row[pred_col].split()
        scores.append(map_at_3(row['answer'], preds))
    return np.mean(scores)

print("MAP@3 ready.")

MAP@3 ready.


# **Dataset**

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df      = df.reset_index(drop=True)
        self.tok     = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        prompt  = str(row['prompt'])
        options = [str(row[c]) for c in OPTION_COLS]

        encodings = []
        for opt in options:
            enc = self.tok(
                prompt, opt,
                max_length=self.max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            encodings.append(enc)

        input_ids      = torch.cat([e['input_ids'] for e in encodings], dim=0)
        attention_mask = torch.cat([e['attention_mask'] for e in encodings], dim=0)
        out = {'input_ids': input_ids, 'attention_mask': attention_mask}

        if not self.is_test:
            out['labels'] = torch.tensor(OPTION_COLS.index(row['answer']), dtype=torch.long)
        return out

train_dataset = MCQDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = MCQDataset(val_df,   tokenizer, MAX_LEN)
test_dataset  = MCQDataset(test,     tokenizer, MAX_LEN, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches: 57
Val batches  : 7


# **Load Model**

In [9]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)

print("Model loaded:", MODEL_NAME)
print("Parameters  :", sum(p.numel() for p in model.parameters()) // 1_000_000, "M")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForMultipleChoice LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilbert-base-uncased
Parameters  : 66 M


# **Training**

In [10]:
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

best_map3  = 0.0
best_epoch = 0
best_val_preds = []

for epoch in range(1, EPOCHS + 1):

    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch} Training"):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        preds    = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        total_loss += loss.item()

    train_acc  = correct / total
    train_loss = total_loss / len(train_loader)

    model.eval()
    val_preds = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch} Val"):
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            logits         = outputs.logits
            top3           = logits.argsort(dim=-1, descending=True)[:, :3]
            for row in top3:
                val_preds.append(' '.join([OPTION_COLS[i] for i in row.cpu().numpy()]))

    val_df['prediction'] = val_preds
    val_map3 = evaluate_map3(val_df)

    print(f"\nEpoch {epoch} | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Val MAP@3: {val_map3:.4f}")

    wandb.log({
        "epoch"      : epoch,
        "train_loss" : train_loss,
        "train_acc"  : train_acc,
        "val_map3"   : val_map3
    })

    if val_map3 > best_map3:
        best_map3      = val_map3
        best_epoch     = epoch
        best_val_preds = val_preds.copy()
        print(f"Best epoch updated: {epoch} | MAP@3: {val_map3:.4f}")

print(f"\nBest Val MAP@3: {best_map3:.4f} at epoch {best_epoch}")

Epoch 1 Training:   0%|          | 0/57 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/




Epoch 1 | Loss: 1.5183 | Acc: 0.3406 | Val MAP@3: 0.7725
Best epoch updated: 1 | MAP@3: 0.7725


Epoch 2 Val: 100%|██████████| 7/7 [00:26<00:00,  3.79s/it]



Epoch 2 | Loss: 0.8843 | Acc: 0.6744 | Val MAP@3: 0.8850
Best epoch updated: 2 | MAP@3: 0.8850


Epoch 3 Val: 100%|██████████| 7/7 [00:25<00:00,  3.59s/it]


Epoch 3 | Loss: 0.5916 | Acc: 0.7789 | Val MAP@3: 0.9233
Best epoch updated: 3 | MAP@3: 0.9233

Best Val MAP@3: 0.9233 at epoch 3


# **Submission**

In [12]:
model.eval()
test_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test Predictions"):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
        logits         = outputs.logits
        top3           = logits.argsort(dim=-1, descending=True)[:, :3]
        for row in top3:
            test_preds.append(' '.join([OPTION_COLS[i] for i in row.cpu().numpy()]))

submission = pd.DataFrame({
    'ID'         : test['id'],
    'Prediction' : test_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission created.")
print(submission.head(10))

wandb.finish()

Test Predictions: 100%|██████████| 16/16 [00:58<00:00,  3.63s/it]

Submission created.
   ID Prediction
0   1      A E B
1   2      E B A
2   3      B E D
3   4      D B E
4   5      C D B
5   6      D B A
6   7      E C D
7   8      B A C
8   9      C D E
9  10      B C E


epoch,▁▅█
train_acc,▁▆█
train_loss,█▃▁
val_map3,▁▆█
epoch,3
train_acc,0.77889
train_loss,0.5916
val_map3,0.92333
